# 1. Import Libraries

In [1]:
import pandas as pd 
import re
from pathlib import Path

# 2. Merge Metadata Files

In [2]:
METADATA_FOLDER = Path.cwd() / 'metadata_list'

metadata_df_list = []

for path_to_file in METADATA_FOLDER.glob('*.csv'):

    print(path_to_file)
    
    sub_metadata_df = pd.read_csv(path_to_file)

    metadata_df_list.append(sub_metadata_df)

metadata_df = pd.concat(metadata_df_list, ignore_index=True)

C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_0-149.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_150-249.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_250-331.csv


In [3]:
for col in ['released_on', 'issued_on', 'adopted']:
    metadata_df[col] = pd.to_datetime(metadata_df[col], errors='coerce')

metadata_df = metadata_df.sort_values(
    'released_on', 
    ascending=False, 
    ignore_index=True
)

In [4]:
metadata_df['download_status'].value_counts()

download_status
downloaded                 3200
skipped_no_matching_txt     111
Name: count, dtype: int64

In [5]:
metadata_df = metadata_df[metadata_df['download_status'] == 'downloaded']

In [6]:
metadata_df['downloaded_filename'] = metadata_df['downloaded_files'].apply(lambda x: Path(str(x)).name)

metadata_df['downloaded_filename_key'] = metadata_df['downloaded_filename'].str.lower()

mask_duplicates = metadata_df['downloaded_filename_key'].duplicated(keep='first')

metadata_df = metadata_df[~mask_duplicates]

In [7]:
metadata_df.to_csv('fcc_news_release_metadata.csv')

# 3. Text Body Extraction

## 3.1. Regular Expressions for News Release Formats

In [8]:
CITY_PREFIX = r'''
    (?:
        \(?\s*
        WASHINGTON
        (?:\s*,?\s*D\.?\s*C\.?)?
        \s*\)?
    )|
    (?:
        BATON\s*ROUGE,\s*La\.|BARCELONA,\s*Spain|BOSTON|BRUSSELS,\s*BELGIUM|BUCHAREST,\s*ROMANIA|CHICAGO|
        HOUSTON|LITTLE\s*ROCK(?:,\s*(?:Ark\.|ARKANSAS))?|LAS\s*VEGAS|LOS\s*ANGELES|MIAMI|
        NEW\s*DELHI,\s*INDIA|OKLAHOMA\s*CITY|PHOENIX|PLUMAS\s*COUNTY,\s*CALIFORNIA|
        SEATTLE|SIOUX\s*FALLS,\s*SOUTH\s*DAKOTA|TAIPEI\s*CITY,\s*TAIWAN|TAMPA|TEXAS
    )
'''

DATELINE = r'''
    \s*\(?        
    [A-Z]{3,9}\.?,?\s+\d{1,2}(?:st|nd|rd|th)?,\s+\d{4}
    \.?\s*\)?\s*
'''

DATELINE_LINE_RE = re.compile(
    rf'''(?ix)
    (?:
        ^\s*(?:[-–—]{{1,2}}|:)?\s*
        (?:
            {CITY_PREFIX}
        )
        \s*,?
        {DATELINE}
        (?:[-–—]{{1,2}}|:)?\s*
    )|
    (?:
        ^{DATELINE}
        (?:[—–]|(?<!-)-(?!-))
        \s*(?=[A-Z])
    )
    '''
)

In [9]:
DATELINE_PREFIX_RE = re.compile(
    r'''(?ix)
    ^\s*
    \(?\s*
    WASHINGTON
    (?:\s*,?\s*D\.?\s*C\.?)?
    \s*\)?
    \s*,?\s*
    (?:[-–—]{1,2}|:)?\s*
    '''
)

In [10]:
DATE_LINE_RE = re.compile(
    r'''(?ix)
    ^
    (?:FOR\s+IMMEDIATE\s+RELEASE:?\s*)?
    (?:[A-Z]{3,9}\.?\s+\d{1,2},\s+\d{4})\b
    '''
)

In [11]:
HEADER_RE = re.compile(
    r'''(?ix)
    ^(?:
        NEWS|
        ADVISORY|
        PUBLIC\s+NOTICE|
        Federal\s+Communications\s+Commission\s*$|
        445\s+12(?:th)?\s+Street|
        th$|
        Street,\s*S\.W\.|
        Washington,?\s*D\.?\s*C\.?\s*20554|
        This\s+is\s+an\s+unofficial\s+announcement|
        constitutes\s+official\s+action|
        See\s+MCI\s+v\.\s*FCC|
        News\s+Media\s+Information|
        Internet:|
        TTY:|
        Fax-On-Demand|
        ftp\.fcc\.gov|
        FOR\s+IMMEDIATE\s+RELEASE|
        NEWS\s+(?:MEDIA\s+)?CONTACTS?:?|
        MEDIA\s+CONTACT:?
    )
    '''
)

In [12]:
CONTACT_RE = re.compile(
    r'''(?ix)
    (
        \bmedia\s+contact\b|
        \bnews\s+contact\b|
        ^contact\b|
        \bemail\b|
        \be-mail\b|
        @fcc\.gov|
        \(?202\)?[- .]?418|
        888[- .]?835
    )
    '''
)

In [13]:
FOOTER_RE = re.compile(
    r'''(?ix)
    ^.*(?:\#\s*){3}\s*$|
    ^\s*[-–—]*\s*FCC\s*(?:News|For)?\s*[-–—]*\s*$|
    ^\s*(?:[A-Z]{2})?\s*Docket\s+No\.?|
    ^\s*for\s*more\s*(?:news|information)?
    '''
)

In [14]:
TITLE_SKIP_RE = re.compile(
    r'''(?ix)
    ^(?:
        [-–—]+|
        \(?[A-Z]{1,4}\s+DOCKET\s+NO\.?|
        Re\s*:
    )
    '''
)

In [15]:
BODY_START_WORDS = r'''
    A|According|ACT|After|As|At|Attorney|Broadband|
    Cable|Chairman|Commissioner|Connect|Consumers|
    During|Ending|Energy|FCC|Federal|For|
    How|I|If|In|Jamie|Joshua|Let|My|Nominations|
    Over|On|Preet|Reflecting|Rep|Secretary|Sen|Senator|
    Taking|Thank|The|This|To|Today|Tomorrow|
    Verizon|Vice|When|While|Wireless|Without
'''

BODY_START_RE = re.compile(
    rf'''(?x)
    ^\s*
    (?i:
        {CITY_PREFIX}
    )
    \s*,?\s*
    (?:[-–—]{{1,2}}|:)?\s*
    ["“”]?
    (?:{BODY_START_WORDS})
    \b
    '''
)

BODY_START_NO_PREFIX_RE = re.compile(
    rf'''(?x)
    ^\s*
    ["“”]?
    (?:{BODY_START_WORDS})
    \b
    '''
)

In [16]:
NORMAL_SENTENCE_LINE_RE = re.compile(
    r'^\s*[A-Z][a-z].*[\.;?!]$'
)

## 3.2. Define Helper Functions

In [17]:
def normalize_lines(text):
    '''
    Convert raw text into clean non-empty lines.
    '''
    text = (
        text.replace('\ufeff', '')
            .replace('\r\n', '\n')
            .replace('\r', '\n')
            .replace('\xa0', ' ')
            .replace('�', '')
            .replace('?', '')
    )

    lines = []

    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            lines.append(line)

    return lines

In [18]:
def upper_ratio(line):
    '''
    Calculate how much of a line is uppercase.
    Useful for detecting title/header lines.
    '''
    letters = [char for char in line if char.isalpha()]

    if not letters:
        return 0

    return sum(char.isupper() for char in letters) / len(letters)

In [19]:
def is_front_matter_line(line):
    '''
    Detect header/title/contact lines before the actual body.
    '''
    if HEADER_RE.search(line):
        return True

    if CONTACT_RE.search(line):
        return True

    if DATE_LINE_RE.search(line):
        return True

    if TITLE_SKIP_RE.search(line):
        return True

    # Many FCC titles are written in uppercase.
    if len(line) <= 180 and upper_ratio(line) >= 0.72 and not line.endswith(('.', '?', '!')):
        return True

    if len(line) <30:
        return True

    return False

In [20]:
def looks_like_body_start(line):
    '''
    Detect a likely first body line.
    '''
    if is_front_matter_line(line):
        return False

    if len(line) < 30:
        return False

    if BODY_START_RE.match(line):
        return True

    return False

In [21]:
def find_body_end(lines):
    '''
    Find the end of the substantive body before footer/contact blocks.
    '''
    for i, line in enumerate(lines):
        if i > len(lines) * 0.51 and FOOTER_RE.match(line):
            return i, 'footer_marker'

    # Some recent releases use trailing media contact blocks.
    for i, line in enumerate(lines):

        if i > len(lines) * 0.37:
            
            if re.match(r'(?i)^\s*(?:Media\s+Contact:|Media\s+Relations:|CGB\s+Contact:)', line):
                return i, 'trailing_media_contact'
    
            if re.match(r'(?i)^\s*Office\s+of\s+Media\s+Relations:', line):
                return i, 'office_media_footer'
    
            if re.match(r'(?i)^\s*Office\s+of\s+Commissioner.+:\s*\(?202\)?', line):
                return i, 'commissioner_office_footer'

            if re.match(r'(?i)^\s*Twitter:?', line):
                return i, 'twitter'

    return len(lines), 'no_footer_marker'

In [22]:
def clean_join(lines):
    '''
    Join wrapped PDF/text lines into one clean text string.
    '''
    body = ' '.join(lines)

    # Remove spaces before punctuation.
    body = re.sub(r'\s+([,.;:?!])', r'\1', body)

    # Collapse repeated spaces.
    body = re.sub(r'\s+', ' ', body).strip()

    return body

In [23]:
def extract_body(text):
    '''
    Extract the main body of one FCC news release.

    Parameters
    ----------
    text : str
        Raw text from one .txt file.

    Returns
    -------
    body_text : str
    extraction_method : str
    footer_method : str
    '''
    lines = normalize_lines(text)

    original_lines = lines

    end_idx, footer_method = find_body_end(lines)
    
    lines = lines[:end_idx]

    # Method 1: Most news releases have a WASHINGTON dateline.
    for i, line in enumerate(lines):
        if DATELINE_LINE_RE.match(line):
            body_lines = lines[i:]

            body_lines[0] = DATELINE_LINE_RE.sub('', body_lines[0]).strip()

            return clean_join(body_lines), 'dateline', footer_method

    # Method 2: If no dateline exists, skip front matter/title/contact lines.
    i = 0

    while i < min(60, len(lines)) and is_front_matter_line(lines[i]):
        i += 1

    for j in range(i, len(lines)):
        if j < len(original_lines) * 0.75 and looks_like_body_start(lines[j]):
            body_lines= lines[j:]

            body_lines[0] = DATELINE_PREFIX_RE.sub('', body_lines[0]).strip()
            
            return clean_join(body_lines), 'fallback_body_like_after_front_matter', footer_method

    # Method 3: Search for body start without city prefix (e.g. Washington)
    for j in range(i, len(lines)):
        if BODY_START_NO_PREFIX_RE.match(lines[j]):
            body_lines= lines[j:]
            
            return clean_join(body_lines), 'fallback_body_like_after_front_matter_no_city_prefix', footer_method

    # Method 4: Search for normal sentence line
    for j in range(i, len(lines)):
        if NORMAL_SENTENCE_LINE_RE.match(lines[j]):
            body_lines= lines[j:]
            
            return clean_join(body_lines), 'fallback_body_like_after_front_matter_normal_sentence', footer_method

    # Method 5: Last fallback.
    for j, line in enumerate(lines):
        if looks_like_body_start(line) or BODY_START_NO_PREFIX_RE.match(line) or NORMAL_SENTENCE_LINE_RE.match(line):
            return clean_join(lines[j:]), 'fallback_body_like_global', footer_method

    return clean_join(lines[i:]), 'fallback_after_front_matter', footer_method

In [24]:
def build_body_dataframe_from_folder(text_folder):
    text_folder = Path(text_folder)

    records = []

    for file_path in sorted(text_folder.glob('*.txt')):
        try:
            raw_text = file_path.read_text(encoding='utf-8-sig')
        except UnicodeDecodeError:
            raw_text = file_path.read_text(encoding='cp1252', errors='replace')

        body_text, extraction_method, footer_method = extract_body(raw_text)

        records.append({
            'filename': file_path.name,
            'body_text': body_text,
            'body_word_count': len(body_text.split()),
            'extraction_method': extraction_method,
            'footer_method': footer_method
        })

    return pd.DataFrame(records)

## 3.3. Collect the Text

In [25]:
TEXT_FOLDER = Path('fcc_news_release_documents')

body_df = build_body_dataframe_from_folder(TEXT_FOLDER)

body_df.head()

,filename,body_text,body_word_count,extraction_method,footer_method
0,100105_Federal_and_State_Partners_To_Test_Nati...,The Department of Homeland Securitys Federal E...,557,fallback_body_like_after_front_matter,footer_marker
1,100105_MEDIA_BUREAU_ANNOUNCES_PANELISTS_AND_AG...,The Media Bureau today announced the panelists...,269,fallback_body_like_after_front_matter,footer_marker
2,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications Commission will hol...,287,fallback_body_like_after_front_matter,footer_marker
3,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"Today, the Federal Communications Commission l...",328,fallback_body_like_after_front_matter,footer_marker
4,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,fallback_body_like_after_front_matter,footer_marker


In [26]:
print('Empty body texts:', (body_df['body_word_count'] == 0).sum())

Empty body texts: 1


In [27]:
body_df.to_csv(
    'fcc_news_release_body_texts.csv',
    index=False,
    encoding='utf-8-sig'
)

In [28]:
body_df['filename_key'] = body_df['filename'].str.lower()

metadata_df = metadata_df.merge(
    body_df,
    left_on='downloaded_filename_key',
    right_on='filename_key',
    how='left'
)

metadata_df = metadata_df.drop(columns=[
    'selected_txt_count', 'download_status', 'error_message',
    'downloaded_files', 'downloaded_filename', 'downloaded_filename_key', 'filename_key'
    ]
)

In [29]:
remove_file_list = [
    '210624_Commissioner_Simington_Participates_On_TIA_Broadband_Panel.txt',
    # FCC-provided original .txt file is encoding-corrupted
    # https://www.fcc.gov/document/commissioner-simington-participates-tia-broadband-panel

    '120807_Spanish_Translation,_PC_Pledge_100_Campaign_Fact_Sheet.txt',
    # Written in Spanish
    # >>> https://www.fcc.gov/document/spanish-translation-pc-pledge-100-campaign-fact-sheet

    '120531_Spanish_Translation,_Fact_Sheet_on_Connect2Compete_Pilot_Program.txt',
    # Written in Spanish
    # >>> https://www.fcc.gov/document/spanish-translation-fact-sheet-connect2compete-pilot-program

    '120412_Broadcast_Station_Totals_as_of_March_31,_2012.txt',
    # Tabular reference list, not a substantive news release
    # >>> https://www.fcc.gov/document/broadcast-station-totals-march-31-2012
    
    '111130_Broadband_Adoption_Presentation_to_FCC_Open_Meeting.txt',
    # Empty .txt file
    # >>> https://www.fcc.gov/document/broadband-adoption-presentation-fcc-open-meeting
    
    '110516_List_Of_Input_Channels_For_Tv_Translators_Stations_As_Of_May_16,_2011.txt',
    # Tabular reference list, not a substantive news release
    # >>> https://www.fcc.gov/document/list-input-channels-tv-translators-stations-may-16-2011

    '110211_BROADCAST_STATION_TOTALS_AS_OF_DECEMBER_31,_2010.txt',
    # Tabular reference list, not a substantive news release
    # >>> https://www.fcc.gov/document/broadcast-station-totals-december-31-2010

    '101130_FCC_IMPLEMENTATION_OF_THE_21ST_CENTURY_COMMUNICATIONS_AND_VIDEO_ACCESSIBILITY_ACT.txt'
    # Presentation slides converted to .txt file
    # >>> https://www.fcc.gov/document/fcc-implementation-21st-century-communications-and-video

]

for url in metadata_df.loc[
    metadata_df['filename'].isin(remove_file_list),
    'webpage_url'
]:
    print(f'>>> {url}')

metadata_df = metadata_df[
    ~metadata_df['filename'].isin(remove_file_list)
]

>>> https://www.fcc.gov/document/commissioner-simington-participates-tia-broadband-panel
>>> https://www.fcc.gov/document/spanish-translation-pc-pledge-100-campaign-fact-sheet
>>> https://www.fcc.gov/document/spanish-translation-fact-sheet-connect2compete-pilot-program
>>> https://www.fcc.gov/document/broadcast-station-totals-march-31-2012
>>> https://www.fcc.gov/document/broadband-adoption-presentation-fcc-open-meeting
>>> https://www.fcc.gov/document/list-input-channels-tv-translators-stations-may-16-2011
>>> https://www.fcc.gov/document/broadcast-station-totals-december-31-2010
>>> https://www.fcc.gov/document/fcc-implementation-21st-century-communications-and-video


In [30]:
metadata_df

,page_title,full_title,document_type,bureaus,description,webpage_url,selected_txt_urls,all_txt_urls,all_txt_count,released_on,issued_on,adopted,filename,body_text,body_word_count,extraction_method,footer_method
0,FCC Proposes to Amend Audible Crawl Rule to Pr...,FCC Proposes to Amend Audible Crawl Rule to Pr...,News Release,Media Relations; Space,NaN,https://www.fcc.gov/document/fcc-proposes-amen...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_to_Amend_Audible_Crawl_Rul...,"Today, the Federal Communications Commission a...",288,dateline,footer_marker
1,FCC Adopts Rules to Enhance the Integrity of t...,FCC Adopts Rules to Enhance the Integrity of t...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-adopts-rules-...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,4,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Adopts_Rules_to_Enhance_the_Integri...,"Today, the Federal Communications Commission a...",342,dateline,footer_marker
2,FCC Targets 'Covered List' Entities' Blanket S...,FCC Proposes Banning Entities on 'Covered List...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-targets-cover...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Covered_List_Entities_Blank...,"Today, the Federal Communications Commission v...",342,dateline,footer_marker
3,FCC Targets Device Test Labs in Nations Withou...,FCC Looks to Prohibit Electronic Device Testin...,News Release,Engineering & Technology; Media Relations,NaN,https://www.fcc.gov/document/fcc-targets-devic...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Device_Test_Labs_in_Nations...,"Today, the Federal Communications Commission v...",406,dateline,footer_marker
4,FCC Proposes Strengthening 'Know-Your-Customer...,FCC Proposes Strengthening 'Know-Your-Customer...,News Release,Consumer and Governmental Affairs; Media Relat...,NaN,https://www.fcc.gov/document/fcc-proposes-stre...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_Strengthening_Know-Your-Cu...,In an effort to stop illegal calls before they...,298,dateline,footer_marker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3191,"Statement of William T. Lake, Chief, Media Bur...","Statement of William T. Lake, Chief, Media Bur...",News Release,Media,NaN,https://www.fcc.gov/document/statement-william...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,fallback_body_like_after_front_matter,footer_marker
3192,FCC Launches Reboot.FCC.gov to Engage Public i...,FCC Launches Reboot.FCC.gov to Engage Public i...,News Release,Office of the Chairman; Media Relations,NaN,https://www.fcc.gov/document/fcc-launches-rebo...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"Today, the Federal Communications Commission l...",328,fallback_body_like_after_front_matter,footer_marker
3193,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,News Release,Wireline Competition,NaN,https://www.fcc.gov/document/panelists-announc...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-06,2010-01-06,NaT,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications 

In [31]:
print('Number of documents:', len(metadata_df))

print('\n', metadata_df['extraction_method'].value_counts())

print('\n', metadata_df['footer_method'].value_counts())

LENGTH_THRESHOLD =50
text_is_short = metadata_df['body_word_count'] < LENGTH_THRESHOLD
print(f'\n Short text (less than {LENGTH_THRESHOLD} words): {text_is_short.sum()}')

Number of documents: 3188

 extraction_method
dateline                                                 2207
fallback_body_like_after_front_matter                     725
fallback_body_like_after_front_matter_no_city_prefix      245
fallback_body_like_after_front_matter_normal_sentence       5
fallback_after_front_matter                                 3
fallback_body_like_global                                   3
Name: count, dtype: int64

 footer_method
footer_marker                 3053
no_footer_marker                83
commissioner_office_footer      23
trailing_media_contact          17
office_media_footer             11
twitter                          1
Name: count, dtype: int64

 Short text (less than 50 words): 9


## 3.4. Inspect the Text

In [32]:
inspect_filter = metadata_df['footer_method'] == 'no_footer_marker'

In [33]:
metadata_df[inspect_filter][['filename', 'body_text', 'extraction_method', 'footer_method']]

,filename,body_text,extraction_method,footer_method
176,250604_Commissioner_Geoffrey_Starks_on_His_Dep...,Federal Communications Commission Commissioner...,dateline,no_footer_marker
177,250604_Commissioner_Simington_Announces_His_De...,Federal Communications Commission Commissioner...,dateline,no_footer_marker
189,250429_Chairman_Carr_Highlights_Wins_Delivered...,"Today, Chairman Carr summarized some of the ke...",dateline,no_footer_marker
333,240807_Telecoms_Respond_to_Chairwoman_Letters_...,Federal Regulatory Relations att.com Washingto...,fallback_body_like_after_front_matter_no_city_...,no_footer_marker
1617,190719_Commissioner_Carr_Staff_Announcements.txt,FCC Commissioner Brendan Carr thanked Jamie Su...,dateline,no_footer_marker
...,...,...,...,...
3099,100511_MOBILE_MINUTES_MADE_SIMPLE_TIPS_FOR_AVO...,"Without automatic usage alerts, it can be hard...",fallback_body_like_after_front_matter_no_city_...,no_footer_marker
3110,100423_FCC_RELEASES_FIRST-EVER_PAPER_ON_ACCESS...,"Today, the Federal Communications Commission i...",fallback_body_like_after_front_matter,no_footer_marker
3126,100402_QUARTERLY_REPORT_ON_INFORMAL_CONSUMER_I...,The Commission has released its report on the ...,fallback_body_like_after_front_matter,no_footer_marker
3178,100121_INTERNATIONAL_BUREAU_ANNOUNCES_NEW_APPO...,"Today, the International Bureau announced four...",fallback_body_like_after_front_matter,no_footer_marker


In [34]:
for filename in metadata_df[inspect_filter]['filename']:
    print(filename)

250604_Commissioner_Geoffrey_Starks_on_His_Departure_From_the_FCC.txt
250604_Commissioner_Simington_Announces_His_Departure.txt
250429_Chairman_Carr_Highlights_Wins_Delivered_During_First_100_Days.txt
240807_Telecoms_Respond_to_Chairwoman_Letters_on_AI_Political_Robocall.txt
190719_Commissioner_Carr_Staff_Announcements.txt
180611_Restoring_Internet_Freedom_Order_Takes_Effect.txt
161110_Clyburn_Invites_Additional_Submissions_for_#Solutions2020_Action_Plan.txt
150504_Chairman_Wheeler,_Commissioner_Pai_Statement_on_Direct_911_Dialing.txt
150413_Statement_of_Commissioner_Pai_on_AM_Radio_Revitalization.txt
150406_FCC_Fines_CenturyLinkIntrado_$17.4M_for_Multi-State_911_Outage.txt
150319_Commissioner_ORielly_Statement_on_FCC_Process_Reform_Task_Force.txt
150317_Statement_of_Commissioner_Michael_ORielly.txt
150312_Chrmn_Wheeler_FCC_Open_Internet_Order_-_Separating_Fact_From_Fiction.txt
150309_Pai_What_People_are_Saying_Post-Adoption_About_Obamas_Internet_Plan.txt
150226_Summary_of_Commissioner

In [35]:
for idx in metadata_df[inspect_filter].index:

    filename = metadata_df.loc[idx, 'filename']
    print(f'<File name {idx}: {filename}> \n')

    file_path = TEXT_FOLDER / filename
    text = file_path.read_text(encoding='utf-8-sig')
    print(f'[Original text] >>> {normalize_lines(text)} \n')
       
    print(f'[Text body] >>> {extract_body(text)} \n')
    print()

<File name 176: 250604_Commissioner_Geoffrey_Starks_on_His_Departure_From_the_FCC.txt> 

[Original text] >>> ['The Honorable Ted Cruz – page 2', 'Media Contact:', 'Justin Faulb, (202) 418-2500', 'Justin.Faulb@fcc.gov', 'For Immediate Release', 'STATEMENT OF COMMISSIONER GEOFFREY STARKS ON HIS DEPARTURE FROM THE FCC', 'WASHINGTON, June 4, 2025—Federal Communications Commission Commissioner Geoffrey Starks issued the following statement:', '“As I previously announced at the May open meeting, my time at the Commission is coming to a close. I will be leaving the Commission at the end of the week, June 6, 2025.', 'Serving as a Commissioner has been the highlight of my career. I am immensely proud of all that we have achieved together. I want to thank my team, and all of the parties that work alongside the Commission – from industry to public sector advocates – for their collaboration and partnership. Most importantly, I want to thank the staff of the Commission – in my opinion, the very bes

In [36]:
comparison_output_path = Path('full_text_vs_cleaned_body.txt')

comparison_blocks = []

for idx in metadata_df.index:

    filename = metadata_df.loc[idx, 'filename']

    file_path = TEXT_FOLDER / filename
    text = file_path.read_text(encoding='utf-8-sig')

    original_lines = normalize_lines(text)
    body_text, extraction_method, footer_method = extract_body(text)

    comparison_block = f'''
============================================================
File index: {idx}
File name: {filename}
Extraction method: {extraction_method}
Footer method: {footer_method}
============================================================

[Original text]

{'\n'.join(original_lines)}

------------------------------------------------------------

[Cleaned text body]

{body_text}

'''

    comparison_blocks.append(comparison_block)


comparison_output_path.write_text(
    '\n'.join(comparison_blocks),
    encoding='utf-8-sig'
)

21472309